train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [2]:
import torch
import torch.nn.functional as F

In [3]:
words = open("data/names.txt", "r").read().splitlines()

In [4]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [6]:
stoi["a"]

1

In [7]:
N = torch.zeros((27,27, 27), dtype=torch.int32)

In [8]:
xs, ys, zs = [], [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)
num = xs.nelement()
print('examples:', num)

examples: 196113


In [9]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)
# gradient descent
for k in range(1050):
  
    # forward pass
    xenc1 = F.one_hot(xs, num_classes=27).float()
    xenc2 = F.one_hot(ys, num_classes=27).float()
    xenc = torch.cat([xenc1, xenc2], dim=1)
    logits = xenc @ W
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
    loss = -probs[torch.arange(num), zs].log().mean() + 0.01*(W**2).mean()
    if k % 50 == 0:
        print(f'{k}:loss={loss.item():.4f}')

    # backward pass
    W.grad = None 
    loss.backward()
    
    # update
    W.data += - 40 * W.grad

0:loss=4.1960
50:loss=2.3154
100:loss=2.2795
150:loss=2.2679
200:loss=2.2624
250:loss=2.2593
300:loss=2.2574
350:loss=2.2562
400:loss=2.2553
450:loss=2.2547
500:loss=2.2542
550:loss=2.2539
600:loss=2.2536
650:loss=2.2534
700:loss=2.2533
750:loss=2.2531
800:loss=2.2530
850:loss=2.2529
900:loss=2.2529
950:loss=2.2528
1000:loss=2.2527


In [10]:
seen = set()
for a in range(60):
    out = []
    ix1 = 0
    ix2 = 0
    while len(out) < 100:
        xenc1 = F.one_hot(torch.tensor([ix1]), num_classes=27).float()
        xenc2 = F.one_hot(torch.tensor([ix2]), num_classes=27).float()
        xenc = torch.cat([xenc1, xenc2], dim=1)
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix3 == 0:
            break
        out.append(itos[ix3])
        ix1 = ix2
        ix2 = ix3
    name = ''.join(out)
    if len(name) > 2 and name not in seen:
        seen.add(name)
        print(name)

aellena
elynncre
arsorie
ahmidon
azayles
asia
izlamayoni
azonce
ene
lann
arayilakhe
oland
nli
usaizad
hia
manoleonyrion
carcelon
syani
lichann
ustnel
rimakaidovielen
ari
aneena
oadely
neryusariegs
ukhle
herizah
amiyanren
aadien
wayahientrikhev
tue
aree
lakviy
utadara
oyah
riaynse
irora
fri
rana
ala
imaram
uan
vyl
savanmaklyanirahasoramiece
astayth
anarlah
